# セグメンテーションデータセット バリデーション — 実行用ノートブック

`README.md` の「3. 手順」をセルに分けて実行できるようにしたもの。
コマンドの意味・出力の読み方・設計判断の理由は README 本体に書いてあるので、
ここでは繰り返さない。**このノートブックはコマンドを打つ手間を減らすためのもの**であり、
判断の根拠が必要になったら都度 README を参照すること。

上から順に実行すれば README と同じ1周ができる。飛ばしたいセルはスキップしてよい。

対応: [README.md](../README.md) 3章 / 参照: [docs/dataset_format.md](../docs/dataset_format.md)

## 0. 前提

- カーネルはこのリポジトリの `uv` 環境（`.venv`）に紐づいていること。
  紐づいていない場合は一度ターミナルで以下を実行してからカーネルを選び直す。

  ```bash
  cd /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation
  uv sync --group notebook --group review   # jupyter + fiftyone 一式
  uv run python -m ipykernel install --user --name segmentation-validation
  ```

- 目視レビュー（3.6〜3.8）まで行うなら `--group review` を付けたままにする
  （外すと fiftyone 一式がアンインストールされる。README 2章）。
- 以降のセルは `!` から始まる**シェルコマンド**（README のコマンドをそのまま実行）と、
  結果を読むための Python セルが混在している。

In [1]:
import os

PROJECT_ROOT = "/mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation"
os.chdir(PROJECT_ROOT)
os.environ.setdefault("FIFTYONE_DATABASE_DIR", os.path.expanduser("~/.fiftyone/var/lib/mongo"))
print("cwd:", os.getcwd())
print("FIFTYONE_DATABASE_DIR:", os.environ["FIFTYONE_DATABASE_DIR"])

cwd: /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation
FIFTYONE_DATABASE_DIR: /home/nakamura/.fiftyone/var/lib/mongo


### 環境確認（任意）

`mongod` のデータ置き場が NFS 上だと壊れる（README 2章）。ローカルディスクか確認する。

In [2]:
!findmnt -T "$HOME" -o TARGET,SOURCE,FSTYPE

TARGET SOURCE   FSTYPE
/      /dev/md0 ext4


In [3]:
# cv2 が1種類だけか（opencv-python と headless が共存すると壊れる）
!uv run python -c "import importlib.metadata as m, cv2; \
  print(cv2.__version__, [d.metadata['Name'] for d in m.distributions() \
  if d.metadata['Name'] and 'opencv' in d.metadata['Name'].lower()])"

4.14.0 ['opencv-python-headless']


### データセットを絞り込む（任意）

特定のデータセットを一時的に除外・限定したいとき（データ側の不具合が見つかった、
特定のデータセットだけで試したい等）は、ここで`CONFIG_OVERRIDES`に書く。
**この1箇所を変えるだけで、以降の全セルに反映される**
（各セルが`SET_ARGS`を`--set ...`として埋め込んでいるため）。
詳しくは [docs/review_procedure.md](../docs/review_procedure.md) 5.1 と README 7章。

In [4]:
# データセットを絞り込みたいときはここに書く。何も絞らないなら空リストのままでよい。
# 例: PTE/PTR/kaggle_pneumothorax混入分を一時除外する（2026-09に実際に使った例）
# CONFIG_OVERRIDES = [
#       'datasets.exclude=["01331","01333","01334","01336","PTE_CX_MT_PI3","PTR_CX_MT_PI3"]'
#   ]
CONFIG_OVERRIDES: list[str] = []

SET_ARGS = " ".join(f"--set '{o}'" for o in CONFIG_OVERRIDES)
print("SET_ARGS:", SET_ARGS or "(なし)")

SET_ARGS: (なし)


## 3.1 対象を確認する

In [5]:
!uv run segmentation-validation {SET_ARGS} list-sources   # 対象JSON

/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01272-20260709_004526.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01331-20260904_051116.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01333-20260904_051143.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01334-20260904_051212.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01336-20260904_051239.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_1298-20260716_105540.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ChestMetry_PI6px_normal-20260903_092412.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_abnormal_non_pneumothorax-20260907_044528.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_pneumothorax_add-20260907_044348.json
/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_with

In [6]:
!uv run segmentation-validation {SET_ARGS} list-checks    # 登録チェック一覧

M01_MASK_RESOLUTION              machine    マスク解像度がDICOMと一致するか  [scan必要]
M02_MASK_CHANNELS                machine    マスクが8bit単チャンネルか  [scan必要]
M03_MASK_BINARY                  machine    背景0・前景255の二値か  [scan必要]
M04_MASK_NOT_EMPTY               machine    空マスク・全塗りでないか  [scan必要]
M05_FILE_EXISTS                  machine    DICOM・マスクの実在と可読性  [scan必要]
M06_PATH_FORMAT                  machine    パス形式の統一
M07_BBOX_GEOMETRY                machine    bbox/elliptical の座標健全性
M08_JSON_MASK_CONSISTENCY        machine    JSONのbbox/region_countと実マスクの一致  [scan必要]
M09_UNANNOTATED_VIEW             machine    アノテーションを持たない画像  [scan必要]
D01_EXACT_DUPLICATE              duplicate  完全一致の重複（同一ラベル）  [scan必要]
D02_EXACT_MASK_LABEL_CONFLICT    duplicate  完全一致だがラベルが異なる  [scan必要]
D03_NEAR_DUPLICATE               duplicate  近似重複  [scan必要]
D04_CONTAINED_DUPLICATE          duplicate  包含された重複  [scan必要]
D05_CROSS_DATASET_DUPLICATE      duplicate  クロスデータセット重複  [scan必要]
S03_TINY_ANNOTATION              suspicious 微小領域  [sca

In [7]:
from segmentation_validation.config import Config
from segmentation_validation.core.cache import fingerprint
print(fingerprint(Config()))


v_701e8639deb8


In [8]:
!uv run segmentation-validation {SET_ARGS} show-config    # 解決後の設定

{
  "project_root": "/mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation",
  "datasets": {
    "sources": [
      "../../../dataset/source/*.json"
    ],
    "exclude": []
  },
  "validation": {
    "target_labels": [
      "pneumothorax"
    ]
  },
  "roots": {
    "image_root": "/mnt",
    "annotation_root": "/mnt/medicaldb"
  },
  "reference_masks": {
    "roots": {
      "lung": "/mnt/medicaldb/processed/lung-mask",
      "thorax": "/mnt/medicaldb/processed/thorax-mask",
      "mediastinum": "/mnt/medicaldb/processed/mediastinum-mask"
    },
    "drop_path_components": [
      0,
      2,
      5,
      6
    ],
    "required": [
      "lung",
      "thorax",
      "mediastinum"
    ],
    "margin_mm": 10.0
  },
  "thresholds": {
    "tiny_annotation_mm2": 20.0,
    "stray_component_mm2": 20.0,
    "stray_component_ratio": 0.01,
    "small_by_class_mm2": {
      "focal": 20.0,
      "localized": 50.0,
      "regional": 150.0
    },
    "small_review_band_factor": 3.0

## 3.2 走査（画素を読む）

**2〜3分**かかる（`--jobs 8`）。2回目以降は走査済みを飛ばして1秒未満で終わる。
共有サーバーなので、走らせる前に他ジョブが張り付いていないか確認しておく。

In [9]:
!cat /proc/loadavg
!uptime

16.13 15.71 16.15 12/16100 1152320
 09:09:43 up 89 days, 18:52, 32 users,  load average: 16.13, 15.71, 16.15


In [10]:
!uv run segmentation-validation {SET_ARGS} scan --jobs 8

INFO 走査済み（47075 ファイル）。やることが無い
INFO 走査したファイル: 0 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/cache/v_701e8639deb8


再走査したいとき（キャッシュ破棄）や、まず少数で試したいときは以下を使う
（普段は不要。コメントアウトのまま残してある）。

```python
# !uv run segmentation-validation {SET_ARGS} scan --jobs 8 --force
# !uv run segmentation-validation {SET_ARGS} scan --jobs 8 --limit 20   # スモークテスト
```

## 3.3 チェックを実行する

→ `issues.json` / `issues.csv`。**error があると exit 1** を返す
（このセルは失敗しても後続を止めたくないので `!` の戻り値は無視している。
 明示的に確認したいときは次のセルの `$?` を見る）。

In [11]:
# 2行に分けて `!` を別々に実行すると $? が引き継がれない（別プロセスになる）ため、
# 1行にまとめて exit code を拾う。
!uv run segmentation-validation {SET_ARGS} check; echo "exit code: $?"

INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO M01_MASK_RESOLUTION                 0 件
INFO M02_MASK_CHANNELS                   0 件
INFO M03_MASK_BINARY                     0 件
INFO M04_MASK_NOT_EMPTY                  0 件
INFO M05_FILE_EXISTS                     0 件
INFO M06_PATH_FORMAT                     2 件
INFO M07_BBOX_GEOMETRY                   0 件
INFO M08_JSON_MASK_CONSISTENCY           1 件
INFO M09_UNANNOTATED_VIEW             36831 件
INFO D01_EXACT_DUPLICATE               668 件
INFO D02_EXACT_MASK_LABEL_CONFLICT      46 件
INFO D03_NEAR_DUPLICATE                 16 件
INFO D04_CONTAINED_DUPLICATE            31 件
INFO D05_CROSS_DATASET_DUPLICATE      6033 件
INFO S03_TINY_ANNOTATION              2921 件
INFO S04_ORIGINAL_FINAL_DIVERGENCE      21 件
INFO S05_OUTSIDE_BODY    

パス規約や bbox の閾値を詰めているときは、JSONだけで判定できるチェックを1秒で回せる。

> **`--only` / `--skip` を使ったら、`select` の前に `--only` なしで回し直すこと。**
> 部分実行のまま `select` すると採否が壊れる（README 3.3 参照）。

In [12]:
# !uv run segmentation-validation {SET_ARGS} check --only M06,M07

## 3.4 採否を確定する

→ `selection_decisions.csv` と `image_decisions.csv`。
初回は `review_decisions.json` が無いので、目視対象は `pending` のままになる（正しい挙動）。

In [13]:
!uv run segmentation-validation {SET_ARGS} select

INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO クロスデータセット重複による自動exclude: 2893 件
INFO 自動採否: exclude 2994 / keep 2975（重複グループ 2975）自動決定不能 2
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=9d4f168e-0b43-4f04-a9f3-4eb3c42ad727
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=32cdebfd-39cd-4da5-b5c1-8515609aa51c
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=f278645a-2804-4c8e-844e-c6f5be76f1c7
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=259b538e-769f-4b92-b11d-c53dbbdbdf91
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用され

## 3.5 結果を見る

`report` で `summary.md`、`gui` で `dashboard.html` を書き出す。
ノートブックなのでコマンドを打つだけでなく、生成された Markdown をそのままここに表示できる。

In [14]:
!uv run segmentation-validation {SET_ARGS} report   # -> summary.md
!uv run segmentation-validation {SET_ARGS} gui      # -> dashboard.html

INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO summary.md -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/summary.md
INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO dashboard.html (19494 KB) -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/dashboard.html
INFO   `serve` で常駐サーバーから開くか、ブラウザで直接開く


In [15]:
import glob
from pathlib import Path

# fingerprint 付きディレクトリのうち最新のものを拾う。
# ディレクトリ自体のmtimeでソートしない: 既存ファイルを上書きするだけの
# report/select はディレクトリのエントリ構成を変えないので、ディレクトリの
# mtimeは更新されないことがある（同名ファイルへの書き込みはOS依存で
# 親ディレクトリのmtimeを変えないことが多い）。中の summary.md 自体のmtimeを見る。
validation_dirs = [Path(p) for p in glob.glob("output/validation/*/")]
latest_validation_dir = max(
    validation_dirs,
    key=lambda p: (p / "summary.md").stat().st_mtime if (p / "summary.md").exists() else 0,
    default=None,
)
print("latest validation dir:", latest_validation_dir)

latest validation dir: output/validation/v_701e8639deb8


In [16]:
from IPython.display import Markdown, display

display(Markdown((latest_validation_dir / "summary.md").read_text()))

# バリデーション要約

- 生成: 2026-09-10T00:10:41.198721+00:00
- fingerprint: `v_701e8639deb8`
- 対象JSON: 12 件
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01272-20260709_004526.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01331-20260904_051116.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01333-20260904_051143.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01334-20260904_051212.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_01336-20260904_051239.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ANN_EIRLPRJ_1298-20260716_105540.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ChestMetry_PI6px_normal-20260903_092412.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_abnormal_non_pneumothorax-20260907_044528.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_pneumothorax_add-20260907_044348.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-ETR_ChestMetry_PI6px_with_mask136-20260624_080014.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-PTE_CX_MT_PI3_pneumothorax-20260904_061537.json`
  - `/mnt/project/chest/metry/pi6/dataset/source/engineer-set-PTR_CX_MT_PI3_pneumothorax-20260904_061507.json`
- 規模: 患者 **46127** / study **47015** / 画像 **47075**
  - annotation あり **10244** 画像（annotation 21388 件）← 検証対象
  - 正常例 **1084** 画像（No Findings。study 全体がアノテーションなし = 意図的な陰性症例）
  - 未アノテーション **35747** 画像（アノテーション済み study の2枚目以降）

## 検出件数

| check_id | 件数 |
|---|---|
| `D01_EXACT_DUPLICATE` | 668 |
| `D02_EXACT_MASK_LABEL_CONFLICT` | 46 |
| `D03_NEAR_DUPLICATE` | 16 |
| `D04_CONTAINED_DIFFERENT_LABEL` | 3 |
| `D04_CONTAINED_DUPLICATE` | 28 |
| `D05_CROSS_DATASET_DUPLICATE` | 5786 |
| `D05_CROSS_DATASET_DUPLICATE_TIE` | 207 |
| `D05_CROSS_DATASET_MISMATCH` | 40 |
| `M06_PATH_LEADING_SLASH` | 2 |
| `M08_JSON_BBOX_MISMATCH` | 1 |
| `M09_NEGATIVE_CASE` | 1084 |
| `M09_UNANNOTATED_SERIES` | 35714 |
| `M09_UNANNOTATED_VIEW` | 33 |
| `S03_STRAY_COMPONENT` | 817 |
| `S03_SUSPICIOUSLY_SMALL` | 2040 |
| `S03_TINY_ANNOTATION` | 64 |
| `S04_ORIGINAL_FINAL_DIVERGENCE` | 21 |
| `S05_OUTSIDE_BODY` | 28 |
| `S05_REFERENCE_UNAVAILABLE` | 3183 |

- severity: error 6544 / info 5310 / warning 37927
- status: cannot_determine 3183 / checked 45514 / not_applicable 1084

## 検査できなかったもの (cannot_determine)

| 理由 | 件数 |
|---|---|
| `cannot_determine:lung_only` | 185 |
| `cannot_determine:no_reference` | 2997 |
| `cannot_determine:reference_corrupt` | 1 |

現在の policy: `cannot_determine_as_review_required = False`
  -> これらは keep のまま。review に回すには設定を `true` にする

## 参照マスクのカバレッジ（毎回実測）

| データセット | ファイル | 3種そろい | annotation | 判定可 |
|---|---|---|---|---|
| ANN_EIRLPRJ_01272 | 296 | 0 (0.0%) | 491 | 0 (0.0%) |
| ANN_EIRLPRJ_01331 | 836 | 0 (0.0%) | 836 | 0 (0.0%) |
| ANN_EIRLPRJ_01333 | 717 | 0 (0.0%) | 717 | 0 (0.0%) |
| ANN_EIRLPRJ_01334 | 825 | 0 (0.0%) | 825 | 0 (0.0%) |
| ANN_EIRLPRJ_01336 | 290 | 0 (0.0%) | 290 | 0 (0.0%) |
| ANN_EIRLPRJ_1298 | 397 | 397 (100.0%) | 898 | 898 (100.0%) |
| ETR_ChestMetry_PI6px_abnormal_non_pneumothorax | 2593 | 2441 (94.1%) | 10364 | 10087 (97.3%) |
| ETR_ChestMetry_PI6px_pneumothorax_add | 2682 | 14 (0.5%) | 2793 | 125 (4.5%) |
| ETR_ChestMetry_PI6px_with_mask136 | 136 | 122 (89.7%) | 276 | 258 (93.5%) |
| PTE_CX_MT_PI3_pneumothorax | 204 | 40 (19.6%) | 271 | 94 (34.7%) |
| PTR_CX_MT_PI3_pneumothorax | 204 | 40 (19.6%) | 271 | 94 (34.7%) |

- ファイル単位の status: available 3054 / cannot_determine:lung_only 394 / cannot_determine:no_reference 5690 / cannot_determine:reference_corrupt 42
- annotation単位の status: available 11556 / cannot_determine:lung_only 457 / cannot_determine:no_reference 5934 / cannot_determine:reference_corrupt 85

## 施設別の検出件数（上位15）

| 施設 | 件数 |
|---|---|
| kaggle_pneumothorax | 20226 |
| todachuo_kenkoukanri | 12238 |
| m3_kurashiki | 5103 |
| tbx11k | 3800 |
| okayama_chuo | 1137 |
| tokyo_medical | 1041 |
| morinomiyako | 1001 |
| asahikawa | 992 |
| eiju_sogo | 927 |
| kajinoki | 824 |
| ofuna_chuo | 477 |
| ishikawa_kenchu | 419 |
| ucc | 300 |
| segmed | 288 |
| fukui_sekijuji | 260 |

## 目視の理由（人間が入れた分）

※ 人間の判定 9 件のうち 9 件は理由が空。App の review_reasons（チェックボックス）で選ぶと集計に乗る。

## アノテータ別（体外領域の warning 以上）

| アノテータ | 件数 |
|---|---|
| kolive23@gmail.com | 10 |
| m.shinzato@hotmail.co.jp | 2 |
| m-shu@email.plala.or.jp | 2 |
| rnhosoi842@gmail.com | 2 |
| tatekawa@viewer.eirl.ai | 1 |
| ryojr01102@gmail.com | 1 |

系統的なアノテータ起因の誤りが疑われる場合はここに偏りが出る。



### dashboard.html を常駐サーバーで開く

ノートブックのセル内に埋め込む（IFrame / data URI）方法も試したが、
1240px 幅で組んである表・グラフをノートブックの狭い出力枠に押し込めると読みにくい。
**実際のブラウザタブで開いたほうが読める。**

**サーバーはこのノートブックから起動しない。** 以前はこのセルがカーネル内に
HTTPサーバーを立てていたが、2つの理由でやめた。

1. サーバーの参照がカーネル内変数にしか無いので、**カーネルを再起動すると
   止められなくなる**（`Address already in use` になり、停止セルも
   `NoneType has no attribute 'shutdown'` で失敗する）。
2. 配信先が**起動時点の**ディレクトリに固定されるので、データセットを足して
   fingerprint が変わると古い成果物を配信し続ける。

代わりに `serve` サブコマンドを**ターミナルから1回上げっぱなし**にする。
配信先はリクエストごとに解決されるので、データセットを足しても再起動は要らない。

```bash
cd /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation

# CONFIG_OVERRIDES と同じ絞り込みを渡す（フル更新のサブプロセスにも引き継がれる）
nohup uv run segmentation-validation serve > output/dashboard_serve.log 2>&1 &

# 止めるとき
pkill -f 'segmentation-validation.*serve'

kill "$(ss -ltnp 'sport = :8899' | grep -oE 'pid=[0-9]+' | head -1 | cut -d= -f2)"
```

ブラウザで <http://localhost:8899/dashboard.html> を開く。
ページ右上に**更新ボタンが2つ**出る（サーバー経由で開いたときだけ表示される）。

| ボタン | 中身 | 目安 |
|---|---|---|
| HTMLを再構成 | ディスク上の `issues.json` / `selection_decisions.json` から作り直す | 数秒 |
| フル更新 | `review export` → `select` → `report` → `gui` を順に実行 | 数分 |

FiftyOne で目視した結果を反映したいときは**フル更新**を押す。
`select` を自分で回したあとなら、ページを再読み込みするだけで
（成果物のほうが新しいことを検知して）作り直される。

ポートは 8899 固定にしてある。README 3.6 の FiftyOne（5151）と同じ理由で、
**手元のSSHトンネル設定を1回書けば毎回使い回せる**ようにするため。
`http://localhost:8899` を開けないときは、本サーバーへのSSH接続にポート転送が
乗っていない（`ssh -L` はSSH接続を新しく張るときにしか付けられない。VS Code Remote の
接続や既に開いているターミナルには後から追加できない）。

**その場しのぎ**: 手元PCで新しいターミナルを開き、以下を実行して接続を張ったままにする
（閉じるとトンネルも切れる）。

```bash
ssh -L 8899:localhost:8899 <本サーバー>
```

**毎回打つのが面倒なら**、手元PCの `~/.ssh/config` に1行足す
（README 3.6 で FiftyOne 用に張っている `Host pi6` ブロックがあれば、そこに追加でよい）。

```
Host pi6
    HostName <本サーバー>
    ProxyJump <踏み台>
    LocalForward 8899 localhost:8899   # dashboard 用（serve）
    LocalForward 5151 localhost:5151   # FiftyOne 用（README 3.6）
```

設定はSSH接続の開始時にしか読まれないので、保存したら**今の接続を閉じて
`ssh pi6` で繋ぎ直す**こと（VS Code Remote ならウィンドウごと再接続）。

In [26]:
# このセルはサーバーを起動しない。上がっているかを見て、リンクか起動コマンドを出すだけ。
# serve は上げっぱなしにするので、ノートブックを開き直しても押し直す必要は無い。
import json
import shlex
import urllib.error
import urllib.request

from IPython.display import Markdown, display

DASHBOARD_PORT = 8899  # SSHトンネル設定を使い回せるように固定（config の gui.port と同値）
DASHBOARD_URL = f"http://localhost:{DASHBOARD_PORT}/dashboard.html"

try:
    with urllib.request.urlopen(
        f"http://127.0.0.1:{DASHBOARD_PORT}/api/status", timeout=3
    ) as response:
        status = json.load(response)
except (urllib.error.URLError, OSError, json.JSONDecodeError) as error:
    start = (
        "nohup uv run segmentation-validation "
        + " ".join(f"--set {shlex.quote(o)}" for o in CONFIG_OVERRIDES)
        + " serve > output/dashboard_serve.log 2>&1 &"
    )
    # 404 が返るのは「別のサーバーがこのポートを掴んでいる」場合。
    # 旧版のノートブックがカーネル内に立てた SimpleHTTPServer が典型で、
    # そのカーネルを落とすまでポートは空かない。
    occupied = isinstance(error, urllib.error.HTTPError)
    note = (
        "**ポート {port} は別のサーバーが掴んでいる**（{error}）。\n\n"
        "旧版のこのノートブックがカーネル内に立てたサーバーである可能性が高い。"
        "`ss -ltnp | grep {port}` で PID を調べ、それが `ipykernel_launcher` なら"
        "**そのカーネルを再起動**する（`serve` はカーネルとは独立なので、"
        "以後この問題は起きない）。\n\n"
        if occupied
        else "**serve が上がっていない**（{error}）。\n\n"
    ).format(port=DASHBOARD_PORT, error=error)
    display(
        Markdown(
            note
            + "プロジェクトルートのターミナルで以下を実行して上げっぱなしにする。\n\n"
            + f"```bash\n{start}\n```\n\n"
            + "上げたらこのセルをもう一度実行するとリンクが出る。"
        )
    )
else:
    served = status["served_fingerprint"]
    lines = [
        f"**[{DASHBOARD_URL}]({DASHBOARD_URL})** をブラウザで開く。",
        "",
        f"- 配信中: `{served}`（選定 {status['served_reason']}）",
        f"- 最終生成: {status['dashboard_mtime']}"
        + ("　**※成果物のほうが新しい。再読み込みで作り直される**" if status["stale"] else ""),
        f"- 人間の判定: annotation {status['review_decisions']['annotations']} 件 / "
        f"画像 {status['review_decisions']['images']} 件",
    ]
    if status["config_fingerprint"] != served:
        lines.append(
            f"- ⚠ このノートブックの設定は `{status['config_fingerprint']}` を指しているが、"
            f"そこに select の成果物が無いので `{served}` を配信している。"
            "`SET_ARGS` が serve 起動時の `--set` と一致しているか確認する。"
        )
    lines += [
        "",
        "開けないときは手元PCの新しいターミナルで以下を実行してから開く",
        "（`~/.ssh/config` に `LocalForward` を書いた場合は不要）。",
        "",
        f"```bash\nssh -L {DASHBOARD_PORT}:localhost:{DASHBOARD_PORT} <本サーバー>\n```",
    ]
    display(Markdown("\n".join(lines)))

**[http://localhost:8899/dashboard.html](http://localhost:8899/dashboard.html)** をブラウザで開く。

- 配信中: `v_701e8639deb8`（選定 config）
- 最終生成: 2026-09-10T00:34:45+00:00　**※成果物のほうが新しい。再読み込みで作り直される**
- 人間の判定: annotation 580 件 / 画像 33 件

開けないときは手元PCの新しいターミナルで以下を実行してから開く
（`~/.ssh/config` に `LocalForward` を書いた場合は不要）。

```bash
ssh -L 8899:localhost:8899 <本サーバー>
```

In [18]:
import pandas as pd

selection_df = pd.read_csv(latest_validation_dir / "selection_decisions.csv")
selection_df["final_decision"].value_counts()

final_decision
keep       16011
exclude     2994
pending     2383
Name: count, dtype: int64

In [19]:
image_df = pd.read_csv(latest_validation_dir / "image_decisions.csv")
image_df["final_decision"].value_counts()

/tmp/ipykernel_1147285/2471863486.py:1: DtypeWarning: Columns (0: reviewer) have mixed types. Specify dtype option on import or set low_memory=False.
  image_df = pd.read_csv(latest_validation_dir / "image_decisions.csv")


final_decision
exclude    35747
keep       11328
Name: count, dtype: int64

## 3.6 FiftyOne で目視する

- `uv sync --group review` 済みであること。
- 初回は25〜30分・約2GBかかる（README 3.6）。**`--all`を既定にしている**
  （`--all`無しで既にkeep/excludeになった画像を除外すると、review buildのたびに
  FiftyOneから消え、その状態で`review export`すると`review_decisions.json`から
  過去の判定ごと失われる事故が実際に起きたため。2回目以降は既存アセットをスキップ
  するので速い）。

In [44]:
!uv run segmentation-validation {SET_ARGS} review build 

INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO 目視対象: annotation 2177 件 / 画像 0 枚 -> 書き出す画像 4287 枚
INFO アセットを書き出す: 4287 画像 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/review（DICOMの全画素を読むので1枚1〜3秒）
INFO 書き出し 19/4287 (0%) 1.8件/秒 残り 39分08秒
INFO 書き出し 25/4287 (1%) 1.8件/秒 残り 38分44秒
INFO 書き出し 46/4287 (1%) 1.9件/秒 残り 36分25秒
INFO 書き出し 50/4287 (1%) 1.9件/秒 残り 36分19秒
INFO 書き出し 69/4287 (2%) 1.9件/秒 残り 36分38秒
INFO 書き出し 75/4287 (2%) 1.9件/秒 残り 37分00秒
INFO 書き出し 99/4287 (2%) 2.0件/秒 残り 35分09秒
INFO 書き出し 100/4287 (2%) 2.0件/秒 残り 35分13秒
INFO 書き出し 121/4287 (3%) 2.0件/秒 残り 34分55秒
INFO 書き出し 125/4287 (3%) 2.0件/秒 残り 34分38秒
INFO 書き出し 144/4287 (3%) 2.0件/秒 残り 34分48秒
INFO 書き出し 150/4287 (3%) 2.0件/秒 残り 34分43秒
INFO 書き出し 169/4287 (4%) 2.0件/秒 残り 34分52秒
INFO 書き出し

In [45]:
# 目視対象だけの軽い build に一時的に戻したいとき（単発。既定は --all）。
# !uv run segmentation-validation {SET_ARGS} review build

### App の起動

`review launch` は `localhost:5151` でサーバーを起動し続けるコマンドなので、
**このセルで直接 `!` 実行すると kernel がブロックされ、他のセルが動かせなくなる。**
バックグラウンドプロセスとして起動し、あとで明示的に止める。

In [20]:
import shlex
import subprocess
import time

review_log = open("output/review_launch.log", "w")
review_proc = subprocess.Popen(
    ["uv", "run", "segmentation-validation", *shlex.split(SET_ARGS), "review", "launch"],
    stdout=review_log,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print("PID:", review_proc.pid, "-> output/review_launch.log を参照")

PID: 1156957 -> output/review_launch.log を参照


手元の端末から SSH トンネルで開く（README 3.6）。

```bash
ssh -L 5151:localhost:5151 <本サーバー>
```

ブラウザで `http://localhost:5151` を開く。**インターネットへ公開しないこと。**

目視の判定入力は README [3.7](../README.md#37-app-での目視のやりかた) を参照
（`review_status` / `review_reason` / `reviewer` を Detection と Sample の
両方に正しく入れる、という部分だけはノートブックでは肩代わりできない）。

目視が終わったら次のセルで App を止める。

In [69]:
review_proc.terminate()
review_proc.wait(timeout=10)
print("stopped:", review_proc.poll())

stopped: 143


## 3.8 判定を採否へ反映する

```
review launch →（目視）→ review export → select → report
```

このループを pending が 0 になるまで繰り返す。

In [78]:
!uv run segmentation-validation {SET_ARGS} review export   # -> review_decisions.json / .csv
!uv run segmentation-validation {SET_ARGS} select           # -> 採否へ反映
!uv run segmentation-validation {SET_ARGS} review status    # 進捗

WARNING reviewer が未入力の判定が 2296 件ある（'unknown' で記録する）
INFO review_decisions: annotation 2422 / 画像 33 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/review/review_decisions.json
INFO 次: `segmentation-validation select` で採否へ反映する
INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
INFO クロスデータセット重複による自動exclude: 2893 件
INFO 自動採否: exclude 3060 / keep 3041（重複グループ 3041）自動決定不能 0
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=9d4f168e-0b43-4f04-a9f3-4eb3c42ad727
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐため無視する: geometry_uid=32cdebfd-39cd-4da5-b5c1-8515609aa51c
WARNING review_decisions.json の annotation行に dataset_id が無い（旧形式）。誤って別データセットへ適用されるのを防ぐ

In [ ]:
!uv run segmentation-validation {SET_ARGS} review precision  # -> precision.md

In [ ]:
display(Markdown((latest_validation_dir / "precision.md").read_text()))

### DB を作り直した後に判定を戻す（round-trip）

`review build` は DB を作り直すコマンドなので、その前に必ず `review export` して
判定を失わないようにする。作り直した後は `review import` で `review_decisions.json`
から判定を復元する。

In [80]:
!uv run segmentation-validation {SET_ARGS} review export    # ★先に必ず export
!uv run segmentation-validation {SET_ARGS} review build --all      # DB 再構築（export 済みなら安全）
!uv run segmentation-validation {SET_ARGS} review import      # review_decisions.json から判定を復元

WARNING reviewer が未入力の判定が 2296 件ある（'unknown' で記録する）
INFO review_decisions: annotation 2422 / 画像 33 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/review/review_decisions.json
INFO 次: `segmentation-validation select` で採否へ反映する
INFO 検証対象 ['pneumothorax'] -> 対象 6817 / 対象外 14571 annotation（対象外はチェックを走らせない）
INFO 対象: 12 データセット / 患者 9715 / study 10221 / 画像 47075（うち annotation あり 10244） / annotation 21388
INFO 計測キャッシュ: 47075 ファイル / 33579 マスク / 48889 ペア
INFO クロスデータセット重複により自動除外: 2893 件
WARNING reviewer が未入力の判定が 2296 件ある（'unknown' で記録する）
INFO 目視対象: annotation 0 件 / 画像 0 枚 -> 書き出す画像 47075 枚（全画像: --all）
WARNING 全 47075 画像を書き出す。DICOMの全画素読みが1枚1〜3秒かかるので初回は25〜30分・約2GB になる（既存アセットはスキップする）
INFO アセットを書き出す: 47075 画像 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/validation/v_701e8639deb8/review（DICOMの全画素を読むので1枚1〜3秒、jobs=8）
INFO 書き出し 25/47075 (0%) 5.1件/秒 残り 2時間34分
INFO 書き出し 50/47075 (0%) 8.6件/秒 残り 1時間30分
INFO 書き出し 75/47075 (0

## 3.9 開発用データセットを生成する

`pending` / `uncertain` が残っていると**既定で止まる**（exit 1）。それが正しい。

In [79]:
!uv run segmentation-validation {SET_ARGS} build-dataset; echo "exit code: $?"

INFO ANN_EIRLPRJ_01272: keep 559 / exclude 0 / 0件になったfile 0 / 画像を落とした 33 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/development/ANN_EIRLPRJ_01272/20260911/development.json
INFO ANN_EIRLPRJ_01331: keep 836 / exclude 0 / 0件になったfile 0 / 画像を落とした 2722 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/development/ANN_EIRLPRJ_01331/20260911/development.json
INFO ANN_EIRLPRJ_01333: keep 717 / exclude 0 / 0件になったfile 0 / 画像を落とした 2840 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/development/ANN_EIRLPRJ_01333/20260911/development.json
INFO ANN_EIRLPRJ_01334: keep 824 / exclude 1 / 0件になったfile 1 / 画像を落とした 2734 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/development/ANN_EIRLPRJ_01334/20260911/development.json
INFO ANN_EIRLPRJ_01336: keep 290 / exclude 0 / 0件になったfile 0 / 画像を落とした 1081 -> /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation/output/development/ANN_EI

先に進める必要がある場合は「許可」と「扱い」の**両方**を明示する
（`--allow-pending` だけではエラーになる）。**通常は使わない — 目視を先に終わらせること。**

In [ ]:
# from datetime import date
# tag = date.today().strftime("%Y%m%d")
# !uv run segmentation-validation {SET_ARGS} build-dataset \
#   --allow-pending --pending-as exclude \
#   --version-tag {tag}

In [ ]:
development_dirs = sorted(
    glob.glob("output/development/*/*/"), key=lambda p: Path(p).stat().st_mtime
)
if development_dirs:
    latest_development_dir = Path(development_dirs[-1])
    print("latest development dir:", latest_development_dir)
    display(Markdown((latest_development_dir.parent / "selection_summary.md").read_text()))
else:
    print("development.json はまだ生成されていない")

## 3.10 単一症例の重畳図（debug 用）

FiftyOne を使わずに1症例だけ図で確認したいとき。ノートブックなら画像をそのままセル出力に表示できる。

In [ ]:
!uv run python -m segmentation_validation.overlay_masks --institution kajinoki

In [ ]:
overlay_pngs = sorted(
    glob.glob("output/overlay/*.png"), key=lambda p: Path(p).stat().st_mtime
)
overlay_pngs[-3:]

In [ ]:
from IPython.display import Image

Image(filename=overlay_pngs[-1]) if overlay_pngs else None

In [ ]:
# 症例やオリジナルマスクを指定したいとき
# !uv run python -m segmentation_validation.overlay_masks --study CXASW00000278_002 --original

## 参考: 設定を変えて回す

閾値やポリシーを変えたら `check` → `select` → `report` → `gui` を回す
（`scan` は不要。計測値は閾値に依存しない）。設定キーの一覧は README 7章。

データセットの絞り込み（`datasets.exclude` 等）は、このノートブックでは
先頭の `CONFIG_OVERRIDES`（3.1節の直前）で一括管理している。ここで都度
`--set` を書く代わりに、そこを編集して以降のセルを再実行すればよい。

In [ ]:
!uv run segmentation-validation {SET_ARGS} show-config